# E-Commerce Sales Data Cleaning & EDA

**Goal:** Clean a messy, realistic e-commerce sales dataset and prepare it for analysis,
simulating real-world data quality issues an analyst would encounter on the job.

**Dataset:** 103 rows of synthetic e-commerce transactions with intentional data quality
issues — missing values, inconsistent formatting, duplicate records, and invalid entries.

**Skills demonstrated:** data validation, deduplication logic, type coercion,
business-rule-based imputation, and documenting data quality decisions.

# import the libraries

In [8]:
import pandas as pd
import numpy as np

# loading the file

In [9]:
df = pd.read_csv("/content/messy_ecommerce_sales_data.csv")

#KNOWING THE DATASET

In [10]:
df.sample(10)

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
21,121,Customer_121,ORD-80067,10/11/2025,Basketball,Sports,4,523.01,Credit Card,Delivered,2092.04
80,180,Customer_180,ORD-86629,3/26/2025,Laptop,NaN,2,418.38,Credit Card,Processing,836.76
60,160,Customer_160,ORD-96037,1/3/2025,Science,Books,5,242.52,Cash on Delivery,Returned,1212.60
42,142,Customer_142,ORD-69018,10/30/2025,Shoes,Clothing,5,645.26,Credit Card,Shipped,2258.41
63,163,Customer_163,ORD-24378,1/5/2023,Smartwatch,Electronics,3,235.26,Cash on Delivery,Shipped,705.78
18,118,Customer_118,ORD-96061,5/31/2025,Science,Books,5,247.82,PayPal,Processing,1239.10
36,136,Customer_136,ORD-20985,6/12/2025,Headphones,NaN,1,696.71,Credit Card,Delivered,696.71
26,126,Customer_126,ORD-11261,5/18/2025,Shoes,Clothing,4,836.8,Bank Transfer,Cancelled,3347.20
73,173,Customer_173,ORD-42171,10/11/2025,Lamp,Home,5,351.89,Credit Card,Shipped,1759.45
11,111,Customer_111,ORD-72349,10/22/2025,Comics,Books,2,865.77,Credit Card,Returned,1731.54


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              103 non-null    int64  
 1    Customer_Name  103 non-null    object 
 2   Order_ID        103 non-null    object 
 3   Order_Date      103 non-null    object 
 4   Product         103 non-null    object 
 5    Category       95 non-null     object 
 6   Quantity        98 non-null     object 
 7   Price           98 non-null     object 
 8   Payment_Method  103 non-null    object 
 9   Status          103 non-null    object 
 10  Total           89 non-null     float64
dtypes: float64(1), int64(1), object(9)
memory usage: 9.0+ KB


In [12]:
df.isnull().sum()

,0
ID,0
Customer_Name,0
Order_ID,0
Order_Date,0
Product,0
Category,8
Quantity,5
Price,5
Payment_Method,0
Status,0


In [13]:
df.duplicated().sum()

np.int64(1)

## Cleaning: fixing column types
`Price` and `Quantity` were stored as text due to invalid entries like `'abd'`,
`'four hundred'`, and `'300$'`. Converted both to numeric using `pd.to_numeric(errors='coerce')`,
which safely turns unparseable junk into null instead of crashing.

In [23]:
# 1. Fix column headers (remove extra spaces)
df.columns = df.columns.str.strip()

# 2. Clean Price column
df['Price'] = df['Price'].astype(str).str.strip()          # remove spaces
df['Price'] = df['Price'].str.replace('$', '', regex=False) # remove $ signs
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')   # turn text into numbers, junk becomes NaN
df['Price'] = df['Price'].round(2)                           # keep 2 decimal places

# 3. Clean Quantity column
df['Quantity'] = df['Quantity'].astype(str).str.strip()
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')  # junk like '4a' becomes NaN

# 4. Check the result
print(df.dtypes)
print(df[['Quantity', 'Price']].head(10))

ID                  int64
Customer_Name      object
Order_ID           object
Order_Date         object
Product            object
Category           object
Quantity          float64
Price             float64
Payment_Method     object
Status             object
Total             float64
dtype: object
   Quantity   Price
0       3.0   38.00
1       2.0     NaN
2       1.0  389.05
3       5.0  233.92
4       1.0  552.51
5       3.0  122.06
6       NaN  978.63
7       5.0  587.68
8       1.0  600.29
9       5.0  168.34



## Resolving duplicate Order_IDs
Some `Order_ID`s appeared twice with conflicting `Total` values. Rather than dropping
arbitrarily, verified which row was correct by checking whether `Total = Quantity × Price`,
and kept the mathematically consistent row.

In [24]:
# Calculate what Total *should* be
df['Calculated_Total'] = df['Quantity'] * df['Price']

# Flag rows where Total matches the calculation (within a small tolerance for floating point)
df['Total_Matches'] = np.isclose(df['Total'], df['Calculated_Total'], atol=0.01)

print("Before removing duplicates:", df.shape)

# Sort so correct rows come first, then keep the first (correct) one per Order_ID
df = df.sort_values('Total_Matches', ascending=False).drop_duplicates(subset='Order_ID', keep='first')
df = df.sort_values('ID').reset_index(drop=True)

print("After removing duplicates:", df.shape)

# Clean up helper columns
df.drop(columns=['Calculated_Total', 'Total_Matches'], inplace=True)

# Sanity check — confirm no more duplicate Order_IDs
print("Remaining duplicate Order_IDs:", df.duplicated(subset='Order_ID').sum())

Before removing duplicates: (103, 13)
After removing duplicates: (100, 13)
Remaining duplicate Order_IDs: 0


## Flagging anomalous negative values
Two rows had negative `Quantity` and `Total`. Their `Status` values (`Processing`, `Shipped`)
didn't support "return" as the explanation, so instead of assuming or deleting, these rows
were flagged with an `Is_Return` column for downstream review — preserving data rather
than guessing at intent.

In [25]:
# Flag negative quantity rows for review, rather than silently keeping or dropping them
df['Is_Return'] = df['Quantity'] < 0

print(df[df['Is_Return']][['Order_ID', 'Quantity', 'Price', 'Total', 'Status', 'Is_Return']])

     Order_ID  Quantity     Price     Total      Status  Is_Return
17  ORD-72751      -2.0  10000.00 -20000.00  Processing       True
34  ORD-16585      -5.0    591.53  -2957.65     Shipped       True



## Standardizing inconsistent categories
`Category` had casing and singular/plural inconsistencies (`electronic`, `ELECTRONICS`,
`Electronics`) that would silently fragment any groupby analysis. Standardized casing,
then merged `Electronic` into `Electronics`.

In [26]:
# Standardize casing first
df['Category'] = df['Category'].str.strip().str.title()

# Check what we have now
print(df['Category'].value_counts(dropna=False))

Category
Books          22
Home           20
Sports         17
Electronics    16
Clothing       14
NaN             8
Electronic      3
Name: count, dtype: int64


In [27]:
# Merge singular 'Electronic' into 'Electronics'
df['Category'] = df['Category'].replace('Electronic', 'Electronics')

print(df['Category'].value_counts(dropna=False))

Category
Books          22
Home           20
Electronics    19
Sports         17
Clothing       14
NaN             8
Name: count, dtype: int64


In [28]:
print(df[df['Category'].isna()][['Product', 'Category']])

       Product Category
33   Biography      NaN
36  Headphones      NaN
80      Laptop      NaN
81  Smartphone      NaN
82       Shoes      NaN
84       Jeans      NaN
93  Basketball      NaN
98      Vacuum      NaN



## Filling missing categories
8 rows had a missing `Category`. Instead of a generic fill, inferred the correct category
from each row's `Product` name using a mapping built from the rest of the dataset —
a more defensible approach than blind imputation.

In [29]:
# Map products to their correct category using a dictionary
product_to_category = {
    'Biography': 'Books',
    'Headphones': 'Electronics',
    'Laptop': 'Electronics',
    'Smartphone': 'Electronics',
    'Shoes': 'Clothing',
    'Jeans': 'Clothing',
    'Basketball': 'Sports',
    'Vacuum': 'Home'
}

# Only fill rows where Category is missing, using the Product name to look it up
df['Category'] = df['Category'].fillna(df['Product'].map(product_to_category))

# Confirm no more missing categories
print(df['Category'].isnull().sum())
print(df['Category'].value_counts())

0
Category
Books          23
Electronics    22
Home           21
Sports         18
Clothing       16
Name: count, dtype: int64


## Parsing inconsistent date formats
`Order_Date` had mixed formats (`M/D/YYYY` and `Jan 5 2023`) plus at least one fully
invalid entry. Used `format='mixed'` to handle both formats in one pass.

In [30]:
# Try converting with mixed format handling
df['Order_Date_Parsed'] = pd.to_datetime(df['Order_Date'], format='mixed', errors='coerce')

# Check how many failed to parse
print("Rows that failed to parse:", df['Order_Date_Parsed'].isna().sum())

# Show the original values that failed
print(df[df['Order_Date_Parsed'].isna()][['Order_ID', 'Order_Date', 'Order_Date_Parsed']])

Rows that failed to parse: 1
     Order_ID Order_Date Order_Date_Parsed
92  ORD-35144        abc               NaT


In [31]:
print(df[df['Order_ID'] == 'ORD-35144'])

     ID Customer_Name   Order_ID Order_Date Product  Category  Quantity  \
92  192  Customer_192  ORD-35144        abc  Jacket  Clothing       NaN   

     Price Payment_Method    Status  Total  Is_Return Order_Date_Parsed  
92  203.63    Credit Card  Returned    NaN      False               NaT  


## Dropping a fully corrupted row
One row (`ORD-35144`) had unparseable garbage in both `Order_Date` and `Quantity` —
multiple broken fields on the same row signals a fully corrupted entry rather than a
single fixable typo, so it was dropped instead of patched.

In [32]:
# Drop the confirmed corrupted row
df = df[df['Order_ID'] != 'ORD-35144'].reset_index(drop=True)

print(df.shape)  # should be 99 rows now

(99, 13)


In [33]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'], format='mixed', errors='coerce')
print(df.dtypes)
print(df.isnull().sum())

ID                            int64
Customer_Name                object
Order_ID                     object
Order_Date           datetime64[ns]
Product                      object
Category                     object
Quantity                    float64
Price                       float64
Payment_Method               object
Status                       object
Total                       float64
Is_Return                      bool
Order_Date_Parsed    datetime64[ns]
dtype: object
ID                    0
Customer_Name         0
Order_ID              0
Order_Date            0
Product               0
Category              0
Quantity              5
Price                 8
Payment_Method        0
Status                0
Total                13
Is_Return             0
Order_Date_Parsed     0
dtype: int64


In [35]:

df.drop(columns=['Order_Date_Parsed'], inplace=True)
print(df.shape)

(99, 12)


## Recovering missing values through calculation
Since `Total = Quantity × Price`, any row missing exactly one of these three values
could have it recalculated from the other two — recovering real data instead of dropping it.

In [36]:
# If Quantity or Price is missing but the other two values are present, we can calculate it
df['Quantity'] = df['Quantity'].fillna(df['Total'] / df['Price'])
df['Price'] = df['Price'].fillna(df['Total'] / df['Quantity'])
df['Total'] = df['Total'].fillna(df['Quantity'] * df['Price'])

print(df.isnull().sum())

ID                 0
Customer_Name      0
Order_ID           0
Order_Date         0
Product            0
Category           0
Quantity           5
Price              8
Payment_Method     0
Status             0
Total             12
Is_Return          0
dtype: int64


In [37]:
print(df[df['Quantity'].isna() | df['Price'].isna() | df['Total'].isna()][['Order_ID', 'Quantity', 'Price', 'Total']])

     Order_ID  Quantity   Price  Total
1   ORD-35783       2.0     NaN    NaN
6   ORD-25885       NaN  978.63    NaN
10  ORD-61020       5.0     NaN    NaN
16  ORD-63660       4.0     NaN    NaN
24  ORD-46136       5.0     NaN    NaN
30  ORD-34007       2.0     NaN    NaN
56  ORD-34679       2.0     NaN    NaN
64  ORD-23010       NaN  587.64    NaN
67  ORD-30329       NaN  593.93    NaN
83  ORD-20916       5.0     NaN    NaN
92  ORD-42475       NaN  522.02    NaN
95  ORD-78384       NaN     NaN    NaN


## Dropping unrecoverable rows
12 rows were missing 2+ of `Quantity`/`Price`/`Total` at once — with two unknowns,
calculation can't recover a unique value, so these were excluded rather than imputed
with unverifiable guesses.

In [38]:
# Drop rows where we can't recover Quantity, Price, or Total
df = df.dropna(subset=['Quantity', 'Price', 'Total']).reset_index(drop=True)

print(df.shape)  # should be 87 rows now (99 - 12)
print(df.isnull().sum())  # should show 0 across the board

(87, 12)
ID                0
Customer_Name     0
Order_ID          0
Order_Date        0
Product           0
Category          0
Quantity          0
Price             0
Payment_Method    0
Status            0
Total             0
Is_Return         0
dtype: int64


In [39]:
df.to_csv('cleaned_ecommerce_sales.csv', index=False)

# Download it from Colab to your machine
from google.colab import files
files.download('cleaned_ecommerce_sales.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Summary
Started with 103 rows containing duplicates, missing values, inconsistent categories,
mixed date formats, and invalid entries. After cleaning:
- Resolved 3 duplicate records using calculated-total verification
- Standardized inconsistent category labels
- Inferred 8 missing categories from product names
- Flagged 2 anomalous negative-value rows instead of silently dropping them
- Removed 1 corrupted row and 12 rows with unrecoverable missing data
- **Final: 87 clean, analysis-ready rows**